<a href="https://colab.research.google.com/github/Mru321/CSI-Cross-Environment-Generalization/blob/main/Temporal%20Sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

RAW_DIR = "/content/drive/MyDrive/Project/Project1/raw_data/renew_dataset/tvt_open_source/raw_traces"
PROCESSED_DIR = "/content/drive/MyDrive/Project/Project1/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
import h5py
import numpy as np
from tqdm import tqdm
import gc

In [ ]:
def load_dataset(file_path):
    f = h5py.File(file_path, 'r')
    data = f['Pilot_Samples']
    return f, data

In [ ]:
def process_file_in_chunks(file_path, save_path, chunk_size=200):

    f, data = load_dataset(file_path)

    num_frames = data.shape[0]

    print("Total frames:", num_frames)

    processed_list = []

    for i in range(0, num_frames, chunk_size):

        chunk = data[i:i+chunk_size]   # ONLY small part loaded

        # ---- Convert to complex ----
        real = chunk[..., 0]
        imag = chunk[..., 1]
        csi = real + 1j * imag

        # ---- Amplitude ----
        amplitude = np.abs(csi).astype(np.float32)

        # ---- Reduce dimension (VERY IMPORTANT) ----
        amplitude = np.mean(amplitude, axis=2)

        processed_list.append(amplitude)

        # Free memory
        del chunk, real, imag, csi, amplitude
        gc.collect()

    f.close()

    # Combine all chunks (SAFE now)
    processed = np.concatenate(processed_list, axis=0)

    # Normalize
    mean = np.mean(processed)
    std = np.std(processed) + 1e-8
    processed = (processed - mean) / std

    np.save(save_path, processed)

    print("Saved:", save_path)

In [ ]:
files = sorted(os.listdir(RAW_DIR))

for file_name in files:

    if not file_name.endswith(".hdf5"):
        continue

    raw_path = os.path.join(RAW_DIR, file_name)
    save_path = os.path.join(PROCESSED_DIR, file_name.replace(".hdf5", ".npy"))

    print(f"\nProcessing: {file_name}")

    process_file_in_chunks(raw_path, save_path)


Processing: channe14_LOS_cluster1.hdf5
Total frames: 8140
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_LOS_cluster1.npy

Processing: channe14_LOS_cluster2.hdf5
Total frames: 8159
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_LOS_cluster2.npy

Processing: channe14_LOS_cluster3.hdf5
Total frames: 8112
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_LOS_cluster3.npy

Processing: channe14_LOS_cluster4.hdf5
Total frames: 8101
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_LOS_cluster4.npy

Processing: channe14_NLOS_cluster1.hdf5
Total frames: 6081
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_NLOS_cluster1.npy

Processing: channe14_NLOS_cluster2.hdf5
Total frames: 4054
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_NLOS_cluster2.npy

Processing: channe14_NLOS_cluster3.hdf5
Total frames: 4058
Saved: /content/drive/MyDrive/Project/Project1/processed/channe14_NLOS_cluster3.npy


In [ ]:
import h5py

file_path = "/content/drive/MyDrive/Project/Project1/raw_data/renew_dataset/tvt_open_source/raw_traces/channe14_NLOS_cluster5.hdf5"

with h5py.File(file_path, 'r') as f:
    print("Keys:", list(f.keys()))

Keys: ['FrameCompleteTime', 'Pilot_Samples', 'RSSI', 'mob_tx_gain']


In [ ]:
with h5py.File(file_path, 'r') as f:
    for key in f.keys():
        data = f[key]
        print(f"\nKey: {key}")
        print("Shape:", data.shape)
        print("Dtype:", data.dtype)


Key: FrameCompleteTime
Shape: (4063, 2)
Dtype: uint64

Key: Pilot_Samples
Shape: (4063, 72, 1120, 2)
Dtype: int16

Key: RSSI
Shape: (4063, 5, 72)
Dtype: uint32

Key: mob_tx_gain
Shape: (104, 4, 2)
Dtype: int32


In [ ]:
with h5py.File(file_path, 'r') as f:
    for key in f.keys():
        print(key, f[key].shape, f[key].dtype)

FrameCompleteTime (4063, 2) uint64
Pilot_Samples (4063, 72, 1120, 2) int16
RSSI (4063, 5, 72) uint32
mob_tx_gain (104, 4, 2) int32


In [ ]:
import os

files = os.listdir(PROCESSED_DIR)

for f in files:
    path = os.path.join(PROCESSED_DIR, f)
    print(f, round(os.path.getsize(path)/1e6, 2), "MB")

csi_feature_dataset.csv 9.17 MB
channe14_LOS_cluster1.npy 2.34 MB
channe14_LOS_cluster2.npy 2.35 MB
channe14_LOS_cluster3.npy 2.34 MB
channe14_LOS_cluster4.npy 2.33 MB
channe14_NLOS_cluster1.npy 1.75 MB
channe14_NLOS_cluster2.npy 1.17 MB
channe14_NLOS_cluster3.npy 1.17 MB
channe14_NLOS_cluster4.npy 1.17 MB
channe14_NLOS_cluster5.npy 1.17 MB


In [ ]:
import numpy as np

data = np.load(os.path.join(PROCESSED_DIR, files[1]), allow_pickle=True)

print("Shape:", data.shape)
print("Dtype:", data.dtype)

Shape: (8140, 72)
Dtype: float32


In [ ]:
import numpy as np
import os

for f in files:
    if not f.endswith(".npy"):
        continue

    path = os.path.join(PROCESSED_DIR, f)
    data = np.load(path)

    print(f)
    print("Shape:", data.shape)
    print("Mean:", np.mean(data))
    print("Std:", np.std(data))
    print("-"*40)